In [ ]:
# ============================================================
# FedProx on Wheat Plant Diseases Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=10,
#             5 clients, Dirichlet α=0.5, μ=0.01
# ============================================================

import os, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    torch.zeros(2,2).to(DEVICE) + 1
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except:
    DEVICE = torch.device("cpu"); print("Using CPU")

NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9
MU              = 0.01

TRAIN_DIR = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/train"
TEST_DIR  = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds    = datasets.ImageFolder(TRAIN_DIR)
NUM_CLASSES = len(train_ds.classes)
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")
all_samples = list(train_ds.samples)

if os.path.exists(TEST_DIR):
    test_ds = datasets.ImageFolder(TEST_DIR)
    remap = {v: train_ds.class_to_idx[k]
             for k,v in test_ds.class_to_idx.items()
             if k in train_ds.class_to_idx}
    for path, lbl in test_ds.samples:
        if lbl in remap: all_samples.append((path, remap[lbl]))
print(f"Total pooled: {len(all_samples)}")

class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        idx = np.where(labels == cls)[0]; np.random.shuffle(idx)
        if len(idx) == 0: continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(idx)).astype(int)
        props[np.argmax(props)] += len(idx) - props.sum()
        for c, split in enumerate(np.split(idx, np.cumsum(props)[:-1])):
            client_indices[c].extend(split.tolist())
    for c in range(num_clients): random.shuffle(client_indices[c])
    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS, DIRICHLET_ALPHA, NUM_CLASSES)
print("\nClient distribution:")
for i, idx in enumerate(client_indices):
    lbls = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | {len(set(lbls))} classes")

def tv_split(indices, val_ratio=0.2):
    random.seed(SEED); indices = list(indices); random.shuffle(indices)
    s = int(len(indices)*(1-val_ratio)); return indices[:s], indices[s:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = tv_split(idx)
    client_train_idx.append(tr); client_val_idx.append(va)

def make_loaders(tr_ids, va_ids):
    tr = DataLoader(SampleDataset([all_samples[i] for i in tr_ids], train_transform),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va = DataLoader(SampleDataset([all_samples[i] for i in va_ids], val_transform),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, va

client_loaders = [make_loaders(client_train_idx[i], client_val_idx[i])
                  for i in range(NUM_CLIENTS)]
all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx], val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Global val size: {len(all_val_idx)}")

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(DEVICE)

global_model = build_model()
print(f"ResNet18 → {NUM_CLASSES} classes")

def fedprox_local_train(local_model, global_params, loader, epochs, lr, momentum, mu):
    local_model.train()
    opt = optim.SGD(local_model.parameters(), lr=lr, momentum=momentum)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad()
            ce = criterion(local_model(imgs), lbls)
            prox = sum(((p - g.detach())**2).sum()
                       for p, g in zip(local_model.parameters(), global_params))
            (ce + (mu/2.0)*prox).backward()
            opt.step()
    return local_model.state_dict()

def fedavg_aggregate(global_model, client_states, client_sizes):
    total = sum(client_sizes)
    avg = copy.deepcopy(client_states[0])
    for k in avg: avg[k] = torch.zeros_like(avg[k], dtype=torch.float32)
    for state, sz in zip(client_states, client_sizes):
        for k in avg: avg[k] += state[k].float() * (sz/total)
    global_model.load_state_dict(avg); return global_model

def evaluate(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            preds.extend(p); labels.extend(lbls.numpy())
    return (accuracy_score(labels, preds),
            precision_score(labels, preds, average='weighted', zero_division=0),
            recall_score(labels, preds, average='weighted', zero_division=0),
            f1_score(labels, preds, average='weighted', zero_division=0))

history = {"round":[],"accuracy":[],"precision":[],"recall":[],"f1":[]}
best_acc, best_state = 0.0, None

print("\n"+"="*65)
print(f"{'FedProx Training — Wheat Plant Diseases':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA} | μ={MU}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS+1):
    global_params = [p.clone().detach() for p in global_model.parameters()]
    states, sizes = [], []
    for c in range(NUM_CLIENTS):
        lm = copy.deepcopy(global_model)
        sd = fedprox_local_train(lm, global_params, client_loaders[c][0],
                                 LOCAL_EPOCHS, LR, MOMENTUM, MU)
        states.append(sd); sizes.append(len(client_train_idx[c]))
    global_model = fedavg_aggregate(global_model, states, sizes)
    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd); history["accuracy"].append(acc)
    history["precision"].append(prec); history["recall"].append(rec)
    history["f1"].append(f1)
    if acc > best_acc: best_acc = acc; best_state = copy.deepcopy(global_model.state_dict())
    print(f"Round {rnd:>2}/{GLOBAL_ROUNDS} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

torch.save(best_state, "fedprox_wheat_best.pth")
print(f"\nBest Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

global_model.load_state_dict(best_state)
fa, fp, fr, ff = evaluate(global_model, global_val_loader)
print(f"Final → Acc={fa:.4f} | Prec={fp:.4f} | Rec={fr:.4f} | F1={ff:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle(f"FedProx — Wheat Plant Diseases (ResNet18)\nClients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, μ={MU}, Rounds={GLOBAL_ROUNDS}", fontsize=13, fontweight='bold')
for ax,(key,label,color) in zip(axes.flatten(),[("accuracy","Accuracy","royalblue"),("precision","Precision","darkorange"),("recall","Recall","green"),("f1","F1-Score","red")]):
    ax.plot(history["round"], history[key], color=color, linewidth=1.8, marker='o', markersize=4)
    ax.set_title(label); ax.set_xlabel("Round"); ax.set_ylabel(label)
    ax.set_xlim(1,GLOBAL_ROUNDS); ax.set_xticks(range(1,GLOBAL_ROUNDS+1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("fedprox_wheat_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n── Per-Client Performance ───────────────────────────────")
for c in range(NUM_CLIENTS):
    acc,prec,rec,f1 = evaluate(global_model, client_loaders[c][1])
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

print("\n── Config ───────────────────────────────────────────────")
for k,v in {"Dataset":"Wheat Plant Diseases","Algorithm":"FedProx","Model":"ResNet18",
            "Clients":NUM_CLIENTS,"Alpha":DIRICHLET_ALPHA,"Mu":MU,"Rounds":GLOBAL_ROUNDS,
            "LocalEpochs":LOCAL_EPOCHS,"LR":LR,"Momentum":MOMENTUM,"BatchSize":BATCH_SIZE,
            "Classes":NUM_CLASSES,"Samples":len(all_samples),
            "BestAcc":f"{best_acc*100:.2f}%","FinalF1":f"{ff:.4f}"}.items():
    print(f"  {k:<16}: {v}")

In [ ]:
# ============================================================
# SCAFFOLD on Wheat Plant Diseases Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=10,
#             5 clients, Dirichlet α=0.5
# SCAFFOLD Option II — global + local control variates
# ============================================================

import os, copy, random, csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    torch.zeros(2,2).to(DEVICE) + 1
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except:
    DEVICE = torch.device("cpu"); print("Using CPU")

NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9

METRICS_CSV = "scaffold_wheat_metrics.csv"

TRAIN_DIR = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/train"
TEST_DIR  = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds    = datasets.ImageFolder(TRAIN_DIR)
NUM_CLASSES = len(train_ds.classes)
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")
all_samples = list(train_ds.samples)

if os.path.exists(TEST_DIR):
    test_ds = datasets.ImageFolder(TEST_DIR)
    remap = {v: train_ds.class_to_idx[k]
             for k,v in test_ds.class_to_idx.items()
             if k in train_ds.class_to_idx}
    for path, lbl in test_ds.samples:
        if lbl in remap: all_samples.append((path, remap[lbl]))
print(f"Total pooled: {len(all_samples)}")

class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        idx = np.where(labels == cls)[0]; np.random.shuffle(idx)
        if len(idx) == 0: continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(idx)).astype(int)
        props[np.argmax(props)] += len(idx) - props.sum()
        for c, split in enumerate(np.split(idx, np.cumsum(props)[:-1])):
            client_indices[c].extend(split.tolist())
    for c in range(num_clients): random.shuffle(client_indices[c])
    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS, DIRICHLET_ALPHA, NUM_CLASSES)
print("\nClient distribution:")
for i, idx in enumerate(client_indices):
    lbls = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | {len(set(lbls))} classes")

def tv_split(indices, val_ratio=0.2):
    random.seed(SEED); indices = list(indices); random.shuffle(indices)
    s = int(len(indices)*(1-val_ratio)); return indices[:s], indices[s:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = tv_split(idx)
    client_train_idx.append(tr); client_val_idx.append(va)

def make_loaders(tr_ids, va_ids):
    tr = DataLoader(SampleDataset([all_samples[i] for i in tr_ids], train_transform),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va = DataLoader(SampleDataset([all_samples[i] for i in va_ids], val_transform),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, va

client_loaders = [make_loaders(client_train_idx[i], client_val_idx[i])
                  for i in range(NUM_CLIENTS)]
all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx], val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Global val size: {len(all_val_idx)}")

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(DEVICE)

global_model = build_model()
print(f"ResNet18 → {NUM_CLASSES} classes")

def zero_like_params(model):
    return [torch.zeros_like(p.data) for p in model.parameters()]

def scaffold_local_train(local_model, global_params, c_global, c_local,
                         loader, epochs, lr):
    # ── FIX: move all reference tensors to DEVICE once ──────────────
    global_params = [p.to(DEVICE) for p in global_params]
    c_global      = [c.to(DEVICE) for c in c_global]
    c_local       = [c.to(DEVICE) for c in c_local]
    # ────────────────────────────────────────────────────────────────

    local_model.train()
    opt = optim.SGD(local_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    K = epochs * len(loader)

    for _ in range(epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad()
            criterion(local_model(imgs), lbls).backward()
            with torch.no_grad():
                for param, ci, c in zip(local_model.parameters(), c_local, c_global):
                    if param.grad is not None:
                        param.grad.data.add_(-ci + c)
            opt.step()

    with torch.no_grad():
        c_local_new, delta_c, delta_y = [], [], []
        for ci, c, wp, wl in zip(c_local, c_global,
                                  global_params, local_model.parameters()):
            ci_new = ci - c + (1.0 / (K * lr)) * (wp - wl.data)
            c_local_new.append(ci_new.cpu())
            delta_c.append((ci_new - ci).cpu())
            delta_y.append((wl.data - wp).cpu())

    return local_model.state_dict(), delta_y, delta_c, c_local_new

def scaffold_aggregate(global_model, client_delta_y, client_delta_c,
                       c_global, client_sizes):
    total = sum(client_sizes)
    with torch.no_grad():
        for i, p in enumerate(global_model.parameters()):
            p.data.add_(sum(client_delta_y[c][i] * (client_sizes[c] / total)
                            for c in range(NUM_CLIENTS)).to(DEVICE))
        for i in range(len(c_global)):
            c_global[i] = (c_global[i].to(DEVICE) +
                           sum(client_delta_c[c][i] for c in range(NUM_CLIENTS))
                           / NUM_CLIENTS).cpu()
    return global_model, c_global

def evaluate(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            preds.extend(p); labels.extend(lbls.numpy())
    return (accuracy_score(labels, preds),
            precision_score(labels, preds, average='weighted', zero_division=0),
            recall_score(labels, preds, average='weighted', zero_division=0),
            f1_score(labels, preds, average='weighted', zero_division=0))

def save_metrics_row(filepath, row: dict):
    """Append one round's metrics to CSV; writes header if file is new."""
    file_exists = os.path.exists(filepath)
    with open(filepath, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

c_global  = zero_like_params(global_model)
c_locals  = [zero_like_params(global_model) for _ in range(NUM_CLIENTS)]

history = {"round":[],"accuracy":[],"precision":[],"recall":[],"f1":[]}
best_acc, best_state = 0.0, None

print("\n"+"="*65)
print(f"{'SCAFFOLD Training — Wheat Plant Diseases':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS+1):
    global_params = [p.data.clone().cpu() for p in global_model.parameters()]
    all_dy, all_dc, sizes = [], [], []

    for c in range(NUM_CLIENTS):
        lm = copy.deepcopy(global_model)
        _, dy, dc, c_new = scaffold_local_train(
            lm, global_params, c_global, c_locals[c],
            client_loaders[c][0], LOCAL_EPOCHS, LR)
        all_dy.append(dy); all_dc.append(dc)
        sizes.append(len(client_train_idx[c]))
        c_locals[c] = c_new

    global_model, c_global = scaffold_aggregate(
        global_model, all_dy, all_dc, c_global, sizes)

    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd); history["accuracy"].append(acc)
    history["precision"].append(prec); history["recall"].append(rec)
    history["f1"].append(f1)

    # ── Save metrics to CSV immediately after each round ────────────
    save_metrics_row(METRICS_CSV, {
        "round": rnd, "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1
    })
    # ────────────────────────────────────────────────────────────────

    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(global_model.state_dict())

    print(f"Round {rnd:>2}/{GLOBAL_ROUNDS} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

# ── Save model weights only once at the end ─────────────────────────
torch.save(best_state, "scaffold_wheat_best.pth")
print(f"\nBest Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Metrics saved to: {METRICS_CSV}")

global_model.load_state_dict(best_state)
fa, fp, fr, ff = evaluate(global_model, global_val_loader)
print(f"Final → Acc={fa:.4f} | Prec={fp:.4f} | Rec={fr:.4f} | F1={ff:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle(f"SCAFFOLD — Wheat Plant Diseases (ResNet18)\nClients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, Rounds={GLOBAL_ROUNDS}", fontsize=13, fontweight='bold')
for ax,(key,label,color) in zip(axes.flatten(),[("accuracy","Accuracy","royalblue"),("precision","Precision","darkorange"),("recall","Recall","green"),("f1","F1-Score","red")]):
    ax.plot(history["round"], history[key], color=color, linewidth=1.8, marker='o', markersize=4)
    ax.set_title(label); ax.set_xlabel("Round"); ax.set_ylabel(label)
    ax.set_xlim(1,GLOBAL_ROUNDS); ax.set_xticks(range(1,GLOBAL_ROUNDS+1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("scaffold_wheat_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n── Per-Client Performance ───────────────────────────────")
for c in range(NUM_CLIENTS):
    acc,prec,rec,f1 = evaluate(global_model, client_loaders[c][1])
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

print("\n── Config ───────────────────────────────────────────────")
for k,v in {"Dataset":"Wheat Plant Diseases","Algorithm":"SCAFFOLD","Model":"ResNet18",
            "Clients":NUM_CLIENTS,"Alpha":DIRICHLET_ALPHA,"Rounds":GLOBAL_ROUNDS,
            "LocalEpochs":LOCAL_EPOCHS,"LR":LR,"Momentum":MOMENTUM,"BatchSize":BATCH_SIZE,
            "Classes":NUM_CLASSES,"Samples":len(all_samples),
            "BestAcc":f"{best_acc*100:.2f}%","FinalF1":f"{ff:.4f}"}.items():
    print(f"  {k:<16}: {v}")

In [ ]:
# FedMA on Wheat Plant Diseases Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=10,
#             5 clients, Dirichlet α=0.5
# FedMA: layer-wise neuron matching via Hungarian algorithm
# ============================================================

import os, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    torch.zeros(2,2).to(DEVICE) + 1
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except:
    DEVICE = torch.device("cpu"); print("Using CPU")

NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9

TRAIN_DIR = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/train"
TEST_DIR  = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds    = datasets.ImageFolder(TRAIN_DIR)
NUM_CLASSES = len(train_ds.classes)
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")
all_samples = list(train_ds.samples)

if os.path.exists(TEST_DIR):
    test_ds = datasets.ImageFolder(TEST_DIR)
    remap = {v: train_ds.class_to_idx[k]
             for k,v in test_ds.class_to_idx.items()
             if k in train_ds.class_to_idx}
    for path, lbl in test_ds.samples:
        if lbl in remap: all_samples.append((path, remap[lbl]))
print(f"Total pooled: {len(all_samples)}")

class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        idx = np.where(labels == cls)[0]; np.random.shuffle(idx)
        if len(idx) == 0: continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(idx)).astype(int)
        props[np.argmax(props)] += len(idx) - props.sum()
        for c, split in enumerate(np.split(idx, np.cumsum(props)[:-1])):
            client_indices[c].extend(split.tolist())
    for c in range(num_clients): random.shuffle(client_indices[c])
    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS, DIRICHLET_ALPHA, NUM_CLASSES)
print("\nClient distribution:")
for i, idx in enumerate(client_indices):
    lbls = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | {len(set(lbls))} classes")

def tv_split(indices, val_ratio=0.2):
    random.seed(SEED); indices = list(indices); random.shuffle(indices)
    s = int(len(indices)*(1-val_ratio)); return indices[:s], indices[s:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = tv_split(idx)
    client_train_idx.append(tr); client_val_idx.append(va)

def make_loaders(tr_ids, va_ids):
    tr = DataLoader(SampleDataset([all_samples[i] for i in tr_ids], train_transform),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va = DataLoader(SampleDataset([all_samples[i] for i in va_ids], val_transform),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, va

client_loaders = [make_loaders(client_train_idx[i], client_val_idx[i])
                  for i in range(NUM_CLIENTS)]
all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx], val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Global val size: {len(all_val_idx)}")

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(DEVICE)

global_model = build_model()
print(f"ResNet18 → {NUM_CLASSES} classes")

def local_train(model, loader, epochs, lr, momentum):
    model.train()
    opt = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad()
            criterion(model(imgs), lbls).backward()
            opt.step()
    return model

def hungarian_match(W_global, W_client):
    Wg = W_global.reshape(W_global.shape[0], -1).cpu().numpy()
    Wc = W_client.reshape(W_client.shape[0], -1).cpu().numpy()
    def norm_rows(M):
        n = np.linalg.norm(M, axis=1, keepdims=True)
        return M / np.where(n==0, 1e-8, n)
    cost = 1.0 - norm_rows(Wg) @ norm_rows(Wc).T
    _, col_ind = linear_sum_assignment(cost)
    return col_ind

def fedma_aggregate(global_model, client_models, client_sizes):
    total      = sum(client_sizes)
    global_sd  = global_model.state_dict()
    client_sds = [m.state_dict() for m in client_models]
    new_sd     = copy.deepcopy(global_sd)

    weight_keys = [k for k in global_sd
                   if k.endswith('.weight') and global_sd[k].dim() >= 2]

    for w_key in weight_keys:
        W_global = global_sd[w_key].float()
        accum    = torch.zeros_like(W_global)
        b_key    = w_key.replace('.weight', '.bias')
        has_bias = b_key in global_sd
        accum_b  = torch.zeros_like(global_sd[b_key].float()) if has_bias else None

        for c_sd, c_size in zip(client_sds, client_sizes):
            w      = c_size / total
            perm   = hungarian_match(W_global, c_sd[w_key].float())
            accum += w * c_sd[w_key].float()[perm]
            if has_bias:
                accum_b += w * c_sd[b_key].float()[perm]

        new_sd[w_key] = accum.to(global_sd[w_key].dtype)
        if has_bias:
            new_sd[b_key] = accum_b.to(global_sd[b_key].dtype)

    for key in global_sd:
        if key in new_sd and not key.endswith('.weight'):
            if key.endswith('.bias') and any(
                    key.replace('.bias','.weight')==w for w in weight_keys):
                continue
            accum = torch.zeros_like(global_sd[key].float())
            for c_sd, c_size in zip(client_sds, client_sizes):
                accum += (c_size/total) * c_sd[key].float()
            new_sd[key] = accum.to(global_sd[key].dtype)

    global_model.load_state_dict(new_sd)
    return global_model

def evaluate(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            preds.extend(p); labels.extend(lbls.numpy())
    return (accuracy_score(labels, preds),
            precision_score(labels, preds, average='weighted', zero_division=0),
            recall_score(labels, preds, average='weighted', zero_division=0),
            f1_score(labels, preds, average='weighted', zero_division=0))

history = {"round":[],"accuracy":[],"precision":[],"recall":[],"f1":[]}
best_acc, best_state = 0.0, None

print("\n"+"="*65)
print(f"{'FedMA Training — Wheat Plant Diseases':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print(f"  Matching: Hungarian algorithm (cosine similarity)")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS+1):
    client_models, sizes = [], []
    for c in range(NUM_CLIENTS):
        lm = copy.deepcopy(global_model)
        lm = local_train(lm, client_loaders[c][0], LOCAL_EPOCHS, LR, MOMENTUM)
        client_models.append(lm.cpu()); sizes.append(len(client_train_idx[c]))

    global_model = global_model.cpu()
    global_model = fedma_aggregate(global_model, client_models, sizes)
    global_model = global_model.to(DEVICE)

    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd); history["accuracy"].append(acc)
    history["precision"].append(prec); history["recall"].append(rec)
    history["f1"].append(f1)
    if acc > best_acc: best_acc = acc; best_state = copy.deepcopy(global_model.state_dict())
    print(f"Round {rnd:>2}/{GLOBAL_ROUNDS} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

torch.save(best_state, "fedma_wheat_best.pth")
print(f"\nBest Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

global_model.load_state_dict(best_state)
fa, fp, fr, ff = evaluate(global_model, global_val_loader)
print(f"Final → Acc={fa:.4f} | Prec={fp:.4f} | Rec={fr:.4f} | F1={ff:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle(f"FedMA — Wheat Plant Diseases (ResNet18)\nClients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, Rounds={GLOBAL_ROUNDS}", fontsize=13, fontweight='bold')
for ax,(key,label,color) in zip(axes.flatten(),[("accuracy","Accuracy","royalblue"),("precision","Precision","darkorange"),("recall","Recall","green"),("f1","F1-Score","red")]):
    ax.plot(history["round"], history[key], color=color, linewidth=1.8, marker='o', markersize=4)
    ax.set_title(label); ax.set_xlabel("Round"); ax.set_ylabel(label)
    ax.set_xlim(1,GLOBAL_ROUNDS); ax.set_xticks(range(1,GLOBAL_ROUNDS+1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("fedma_wheat_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n── Per-Client Performance ───────────────────────────────")
for c in range(NUM_CLIENTS):
    acc,prec,rec,f1 = evaluate(global_model, client_loaders[c][1])
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

print("\n── Config ───────────────────────────────────────────────")
for k,v in {"Dataset":"Wheat Plant Diseases","Algorithm":"FedMA","Model":"ResNet18",
            "Matching":"Hungarian (cosine)","Clients":NUM_CLIENTS,"Alpha":DIRICHLET_ALPHA,
            "Rounds":GLOBAL_ROUNDS,"LocalEpochs":LOCAL_EPOCHS,"LR":LR,
            "Momentum":MOMENTUM,"BatchSize":BATCH_SIZE,"Classes":NUM_CLASSES,
            "Samples":len(all_samples),"BestAcc":f"{best_acc*100:.2f}%",
            "FinalF1":f"{ff:.4f}"}.items():
    print(f"  {k:<16}: {v}")

In [ ]:
# ============================================================
# FedAvg on Wheat Plant Diseases Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=10,
#             5 clients, Dirichlet α=0.5
# Dataset: https://www.kaggle.com/datasets/kushagra3204/wheat-plant-diseases
# ============================================================

# ── Cell 0: Imports ──────────────────────────────────────────
import os, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── Cell 1: Reproducibility & Device ────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    torch.zeros(2,2).to(DEVICE) + 1
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except:
    DEVICE = torch.device("cpu")
    print("Using CPU")

# ── Cell 2: Hyperparameters ──────────────────────────────────
NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9

TRAIN_DIR = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/train"
TEST_DIR  = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/test"

# ── Cell 3: Transforms ───────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Cell 4: Load & pool ──────────────────────────────────────
train_ds    = datasets.ImageFolder(TRAIN_DIR)
NUM_CLASSES = len(train_ds.classes)
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")
all_samples = list(train_ds.samples)

if os.path.exists(TEST_DIR):
    test_ds = datasets.ImageFolder(TEST_DIR)
    remap = {v: train_ds.class_to_idx[k]
             for k,v in test_ds.class_to_idx.items()
             if k in train_ds.class_to_idx}
    for path, lbl in test_ds.samples:
        if lbl in remap:
            all_samples.append((path, remap[lbl]))

print(f"Total pooled: {len(all_samples)}")

# ── Cell 5: Dataset class ────────────────────────────────────
class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

# ── Cell 6: Dirichlet split ──────────────────────────────────
def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        idx = np.where(labels == cls)[0]
        np.random.shuffle(idx)
        if len(idx) == 0: continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(idx)).astype(int)
        props[np.argmax(props)] += len(idx) - props.sum()
        for c, split in enumerate(np.split(idx, np.cumsum(props)[:-1])):
            client_indices[c].extend(split.tolist())
    for c in range(num_clients): random.shuffle(client_indices[c])
    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS,
                                 DIRICHLET_ALPHA, NUM_CLASSES)
print("\nClient distribution:")
for i, idx in enumerate(client_indices):
    lbls = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | {len(set(lbls))} classes")

# ── Cell 7: Train/val split & loaders ───────────────────────
def tv_split(indices, val_ratio=0.2):
    random.seed(SEED); indices = list(indices); random.shuffle(indices)
    s = int(len(indices)*(1-val_ratio))
    return indices[:s], indices[s:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = tv_split(idx)
    client_train_idx.append(tr); client_val_idx.append(va)

def make_loaders(tr_ids, va_ids):
    tr = DataLoader(SampleDataset([all_samples[i] for i in tr_ids], train_transform),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va = DataLoader(SampleDataset([all_samples[i] for i in va_ids], val_transform),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, va

client_loaders = [make_loaders(client_train_idx[i], client_val_idx[i])
                  for i in range(NUM_CLIENTS)]

all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx], val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"\nGlobal val size: {len(all_val_idx)}")

# ── Cell 8: Model ────────────────────────────────────────────
def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(DEVICE)

global_model = build_model()
print(f"ResNet18 → {NUM_CLASSES} classes")

# ── Cell 9: Local train ──────────────────────────────────────
def local_train(model, loader, epochs, lr, momentum):
    model.train()
    opt = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward(); opt.step()
    return model.state_dict()

# ── Cell 10: FedAvg aggregation ──────────────────────────────
def fedavg_aggregate(global_model, client_states, client_sizes):
    total = sum(client_sizes)
    avg   = copy.deepcopy(client_states[0])
    for k in avg: avg[k] = torch.zeros_like(avg[k], dtype=torch.float32)
    for state, sz in zip(client_states, client_sizes):
        for k in avg: avg[k] += state[k].float() * (sz/total)
    global_model.load_state_dict(avg); return global_model

# ── Cell 11: Evaluate ────────────────────────────────────────
def evaluate(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            preds.extend(p); labels.extend(lbls.numpy())
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(labels, preds, average='weighted', zero_division=0)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    return acc, prec, rec, f1

# ── Cell 12: Training loop ───────────────────────────────────
history = {"round":[],"accuracy":[],"precision":[],"recall":[],"f1":[]}
best_acc, best_state = 0.0, None

print("\n"+"="*65)
print(f"{'FedAvg Training — Wheat Plant Diseases':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS+1):
    states, sizes = [], []
    for c in range(NUM_CLIENTS):
        lm = copy.deepcopy(global_model)
        sd = local_train(lm, client_loaders[c][0], LOCAL_EPOCHS, LR, MOMENTUM)
        states.append(sd); sizes.append(len(client_train_idx[c]))
    global_model = fedavg_aggregate(global_model, states, sizes)
    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd); history["accuracy"].append(acc)
    history["precision"].append(prec); history["recall"].append(rec)
    history["f1"].append(f1)
    if acc > best_acc: best_acc = acc; best_state = copy.deepcopy(global_model.state_dict())
    print(f"Round {rnd:>2}/{GLOBAL_ROUNDS} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

torch.save(best_state, "fedavg_wheat_best.pth")
print(f"\nBest Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ── Cell 13: Final eval ──────────────────────────────────────
global_model.load_state_dict(best_state)
fa, fp, fr, ff = evaluate(global_model, global_val_loader)
print(f"\nFinal → Acc={fa:.4f} | Prec={fp:.4f} | Rec={fr:.4f} | F1={ff:.4f}")

# ── Cell 14: Plot ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle(f"FedAvg — Wheat Plant Diseases (ResNet18)\nClients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, Rounds={GLOBAL_ROUNDS}", fontsize=13, fontweight='bold')
for ax,(key,label,color) in zip(axes.flatten(),[("accuracy","Accuracy","royalblue"),("precision","Precision","darkorange"),("recall","Recall","green"),("f1","F1-Score","red")]):
    ax.plot(history["round"], history[key], color=color, linewidth=1.8, marker='o', markersize=4)
    ax.set_title(label); ax.set_xlabel("Round"); ax.set_ylabel(label)
    ax.set_xlim(1,GLOBAL_ROUNDS); ax.set_xticks(range(1,GLOBAL_ROUNDS+1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("fedavg_wheat_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Cell 15: Per-client & config ─────────────────────────────
print("\n── Per-Client Performance ───────────────────────────────")
for c in range(NUM_CLIENTS):
    acc,prec,rec,f1 = evaluate(global_model, client_loaders[c][1])
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

print("\n── Config ───────────────────────────────────────────────")
for k,v in {"Dataset":"Wheat Plant Diseases","Algorithm":"FedAvg","Model":"ResNet18",
            "Clients":NUM_CLIENTS,"Alpha":DIRICHLET_ALPHA,"Rounds":GLOBAL_ROUNDS,
            "LocalEpochs":LOCAL_EPOCHS,"LR":LR,"Momentum":MOMENTUM,
            "BatchSize":BATCH_SIZE,"Classes":NUM_CLASSES,"Samples":len(all_samples),
            "BestAcc":f"{best_acc*100:.2f}%","FinalF1":f"{ff:.4f}"}.items():
    print(f"  {k:<16}: {v}")